# Bài học 10 - Tác nhân AI trong sản xuất

Trong bài học này, bạn sẽ học các **mẫu sản xuất** cho các tác nhân AI sử dụng Microsoft Agent Framework (Python). Chúng tôi bao gồm:

- **Khả năng quan sát** — thêm thời gian và ghi nhật ký vào các tương tác của tác nhân
- **Đánh giá** — sử dụng tác nhân đánh giá để chấm điểm chất lượng phản hồi
- **Quản lý chi phí** — các chiến lược tối ưu hóa token và lựa chọn mô hình

Kịch bản là một **đại lý du lịch** giúp người dùng lên kế hoạch cho các chuyến đi, với việc giám sát và đánh giá được xếp chồng lên trên.


## Thiết lập


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Những Cân Nhắc Khi Đưa Vào Sản Xuất

Việc chuyển các đại lý trí tuệ nhân tạo từ bản mẫu sang sản xuất đòi hỏi sự chú ý cẩn thận đến ba trụ cột:

1. **Khả năng quan sát** — Bạn cần biết đại lý đang làm gì, mất bao lâu và công cụ nào được gọi. Nếu không có theo dõi và ghi nhật ký, việc gỡ lỗi các vấn đề trong sản xuất gần như không thể.

2. **Đánh giá** — Các kiểm tra chất lượng tự động đảm bảo phản hồi của đại lý vẫn chính xác, đầy đủ và hữu ích theo thời gian. Một đại lý đánh giá có thể chấm điểm phản hồi dựa trên các tiêu chí đã định nghĩa.

3. **Quản lý chi phí** — Việc sử dụng token ảnh hưởng trực tiếp đến chi phí. Các chiến lược như tối ưu hoá prompt, lựa chọn mô hình và bộ nhớ đệm giúp kiểm soát chi phí mà không làm giảm chất lượng.


## Xây dựng một Đại lý Quan sát được

Chúng tôi định nghĩa các công cụ du lịch và bao bọc cuộc gọi đại lý với tính năng đo thời gian để chúng tôi có thể giám sát độ trễ. Trong môi trường sản xuất, bạn sẽ tích hợp với OpenTelemetry hoặc một backend theo dõi tương tự.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mẫu Đánh Giá

Một mẫu phổ biến trong sản xuất là sử dụng một tác nhân thứ hai làm **người đánh giá**. Người đánh giá chấm điểm phản hồi của tác nhân chính dựa trên các tiêu chí định trước như độ hoàn chỉnh, độ chính xác và tính hữu ích.

Điều này cho phép:
- Cổng chất lượng tự động trước khi phản hồi đến người dùng
- Phát hiện thoái lui khi lời nhắc hoặc mô hình thay đổi
- Giám sát liên tục hiệu suất của tác nhân theo thời gian


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Chiến lược quản lý chi phí

Kiểm soát chi phí rất quan trọng đối với các tác nhân AI sản xuất. Dưới đây là các chiến lược chính:

| Chiến lược | Mô tả |
|---|---|
| **Tối ưu hóa prompt** | Giữ hướng dẫn hệ thống ngắn gọn. Loại bỏ ngữ cảnh dư thừa để giảm số lượng token đầu vào. |
    "| **Lựa chọn mô hình** | Sử dụng các mô hình nhỏ hơn, rẻ hơn (ví dụ GPT-4.1-mini) cho các tác vụ đơn giản như phân loại hoặc trích xuất, và dành riêng các mô hình lớn hơn cho suy luận phức tạp. |\n",
| **Lưu trong bộ nhớ đệm** | Lưu kết quả công cụ và các truy vấn thường xuyên để tránh gọi API lặp đi lặp lại. |
| **Ngân sách token** | Đặt giới hạn `max_tokens` để ngăn phản hồi dài bất ngờ. |
| **Gộp lệnh** | Gom nhiều truy vấn của người dùng vào một lần gọi API khi có thể. |

Trong thực tế, cách tiếp cận theo cấp bậc hoạt động tốt: chuyển các yêu cầu đơn giản đến mô hình nhanh và rẻ tiền, chỉ nâng cấp các truy vấn phức tạp lên mô hình có khả năng cao hơn.


## Tóm tắt

Trong bài học này bạn đã học cách:

1. **Thêm khả năng quan sát** vào các tương tác của đại lý với thời gian và ghi nhật ký, tạo nền tảng cho việc theo dõi và giám sát.
2. **Đánh giá phản hồi của đại lý** tự động bằng cách sử dụng đại lý đánh giá chấm điểm về tính đầy đủ, chính xác và hữu ích.
3. **Quản lý chi phí** thông qua tối ưu hóa yêu cầu, lựa chọn mô hình, lưu bộ nhớ đệm và phân bổ ngân sách token.

Các mẫu sản xuất này giúp đảm bảo các đại lý AI của bạn đáng tin cậy, có thể đo lường và tiết kiệm chi phí ở quy mô lớn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
